# Lab6.3 — Real-Time Video Processing and End-to-End Benchmark

**Edge application:** read a video frame, run Resize → Grayscale → Sobel, derive an edge-activity indicator, repeat. Compare two **actually available** execution paths:
- **A / ARM:** all OpenCV on ARM;
- **B / hybrid:** Resize in PL via the Lab5 overlay, remaining stages on ARM.

The future fused Resize → Sobel FPGA pipeline was **designed in Lab6.2 but not yet provided as a verified bit/hwh overlay**. We do not substitute an imaginary speed-up or simulated PL timing.

**Deliverables:** acquisition/processing/end-to-end mean and p95, measured whole-loop FPS, a visual result, identification of bottlenecks, and one documented parameter experiment.

## 1. Source selection and experimental protocol

Set MODE to **synthetic** (works offline, no camera), **file** (local video), or **camera** (USB, e.g. Logitech C270). The synthetic source is a moving image created in software; its generation time **must not be presented as camera acquisition time**.

Each path opens a fresh source. Synthetic frames and file frames are reproducible, but two sequential USB camera runs will not capture identical real-world frames.

We will report **measured timing** for the selected actual source, and distinguish it from hardware theoretical throughput.

In [ ]:
from pathlib import Path
from time import perf_counter
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lab6_common import (load_test_image, locate_resizer, ResizerHW,
                         process_cpu, process_hybrid)

MODE = "synthetic"        # "synthetic", "file", or "camera"
VIDEO_FILE = Path("video.mp4")  # used only for MODE="file"
CAMERA_INDEX = 0
N_FRAMES = 50
TARGET = (320, 240)
EDGE_THRESHOLD = 80

class VideoSource:
    def __init__(self, mode):
        self.mode = mode
        self.index = 0
        self.cap = None
        if mode == "synthetic":
            self.base = cv2.resize(load_test_image(), (640, 480),
                                   interpolation=cv2.INTER_LINEAR)
        elif mode in ("file", "camera"):
            if mode == "file" and not VIDEO_FILE.is_file():
                raise FileNotFoundError("Missing video file: " + str(VIDEO_FILE))
            self.cap = cv2.VideoCapture(str(VIDEO_FILE) if mode == "file"
                                        else CAMERA_INDEX)
            if not self.cap.isOpened():
                raise RuntimeError("Cannot open %s source." % mode)
            if mode == "camera":
                self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
                self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        else:
            raise ValueError("MODE must be synthetic, file, or camera")

    def read(self):
        if self.mode == "synthetic":
            h, w = self.base.shape[:2]
            dx = (self.index * 7) % 51 - 25
            dy = (self.index * 3) % 31 - 15
            M = np.float32([[1, 0, dx], [0, 1, dy]])
            frame = cv2.warpAffine(self.base, M, (w, h),
                                   borderMode=cv2.BORDER_REFLECT_101)
            self.index += 1
            return True, frame
        ok, frame = self.cap.read()
        if not ok and self.mode == "file":
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ok, frame = self.cap.read()
        return ok, frame

    def close(self):
        if self.cap is not None:
            self.cap.release()

print("Source:", MODE, "Frames:", N_FRAMES, "Output:", TARGET)

## 2. Execute a full application loop

We measure:
- **source time**: USB/file acquisition or synthetic image generation;
- **processing time**: Resize, Grayscale and Sobel along the selected path;
- **decision time**: computing the proportion of strong edge pixels;
- **end-to-end per frame**: everything from starting acquisition to producing the decision;
- **whole-loop FPS**: number of successful frames divided by actual elapsed wall time.

No per-frame plotting is performed inside the timed loop. Displaying video or sending it through a notebook browser would add another transport/rendering cost not covered by this experiment.

In [ ]:
def run_application(path, processor, n_frames=N_FRAMES):
    source = VideoSource(MODE)
    samples = []
    preview = None
    t_start = perf_counter()
    try:
        for _ in range(n_frames):
            t0 = perf_counter()
            ok, frame = source.read()
            t1 = perf_counter()
            if not ok:
                break
            edges = processor(frame)
            t2 = perf_counter()
            edge_fraction = float(np.mean(edges >= EDGE_THRESHOLD))
            t3 = perf_counter()
            if preview is None:
                preview = (frame.copy(), edges.copy())
            samples.append((1000*(t1-t0), 1000*(t2-t1),
                            1000*(t3-t2), 1000*(t3-t0), edge_fraction))
    finally:
        source.close()
    total_seconds = perf_counter() - t_start
    if not samples:
        raise RuntimeError("No usable frames from source " + MODE)
    data = np.asarray(samples)
    report = dict(path=path, count=len(samples),
                  source_mean_ms=float(np.mean(data[:, 0])),
                  process_mean_ms=float(np.mean(data[:, 1])),
                  decision_mean_ms=float(np.mean(data[:, 2])),
                  e2e_mean_ms=float(np.mean(data[:, 3])),
                  e2e_p95_ms=float(np.percentile(data[:, 3], 95)),
                  observed_loop_fps=float(len(samples)/total_seconds),
                  edge_fraction_mean=float(np.mean(data[:, 4])))
    return report, preview

arm_report, arm_preview = run_application(
    "A / ARM", lambda frame: process_cpu(frame, TARGET))
print("A:", arm_report)

## 3. Run the hybrid path, when a real Resize overlay is available

The hybrid path uses the **same functional task** and video source. FPGA initialization is excluded from the loop (typical steady-state deployment); individual buffer reconfigurations triggered by unexpected camera resolution changes are not hidden.

If hardware is unavailable, leave the row B empty and explain why. **Do not fabricate a PL FPS.**

In [ ]:
hw = None
hybrid_report = None
hybrid_preview = None
try:
    bit = locate_resizer()
    hw = ResizerHW(bit)
    print("Hybrid bitfile:", bit)
    hybrid_report, hybrid_preview = run_application(
        "B / hybrid", lambda frame: process_hybrid(frame, TARGET, hw))
    print("B:", hybrid_report)
except (ImportError, FileNotFoundError, AttributeError, RuntimeError) as exc:
    print("B not measured (missing/incompatible hardware or video source):", exc)
finally:
    if hw is not None:
        hw.close()
        hw = None

reports = [arm_report] + ([hybrid_report] if hybrid_report else [])
print("\n%-14s %11s %11s %11s %11s %11s" %
      ("Path", "Source ms", "Proc. ms", "Decision ms", "E2E ms", "FPS loop"))
for r in reports:
    print("%-14s %11.2f %11.2f %11.2f %11.2f %11.2f" %
          (r["path"], r["source_mean_ms"], r["process_mean_ms"],
           r["decision_mean_ms"], r["e2e_mean_ms"], r["observed_loop_fps"]))

## 4. Visualization and interpretation

A high PL kernel throughput alone does not imply the same application FPS. The camera driver, frame generation, Python processing, DMA and the decision stage can all become bottlenecks.

For synchronous, sequential loops, the achieved FPS also reflects acquisition pacing and software overhead. Mean frame latency and reciprocal mean latency are **not identical** to a universal maximum processing throughput; report the actual whole-loop FPS separately.

In [ ]:
preview = hybrid_preview if hybrid_preview is not None else arm_preview
src, edges = preview
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(cv2.cvtColor(src, cv2.COLOR_BGR2RGB))
axes[0].set_title("Captured/generated frame")
axes[1].imshow(edges, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Resize + Sobel edges")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

x = np.arange(len(reports))
width = 0.22
plt.figure(figsize=(8, 4))
for offset, key, name in [(-width, "source_mean_ms", "Source"),
                          (0, "process_mean_ms", "Processing"),
                          (width, "decision_mean_ms", "Decision")]:
    plt.bar(x+offset, [r[key] for r in reports], width=width, label=name)
plt.xticks(x, [r["path"] for r in reports])
plt.ylabel("Measured mean ms per frame")
plt.title("Application timing breakdown / source=%s" % MODE)
plt.legend(); plt.tight_layout(); plt.show()

## 5. Integrated investigation: when does acceleration matter?

Repeat this notebook with **two output resolutions** (for example 160×120 and 480×320) and, if available, **two actual input sources**. Record source type, actual frame dimensions, frame count and mode.

Your report should answer:

1. What fraction of E2E is spent acquiring the frame vs processing it? Is acquisition the bottleneck?
2. Does the hybrid route increase measured FPS for this use case? Support the claim with actual numbers.
3. Explain why the synthetic-source results cannot be reported as USB-camera throughput.
4. For a fused Resize → Sobel FPGA overlay, *which* measured cost could plausibly fall? Which costs would remain?
5. Propose one realistic edge application for which reduced DDR traffic matters more than peak kernel arithmetic speed (e.g. concurrent sensing workloads).

**A/B/C status for the final report:** A and B are measured when supported; C, the fused FPGA streaming pipeline, is an architecture studied in Lab6.2, not a measured implementation.